# Reproducing SparseGPT Results

This notebook helps reproduce the sparsity-vs-perplexity comparison from the SparseGPT paper.

The original paper tests on **OPT-175B**, but we'll demonstrate with smaller models that are more accessible.

## 1. Setup and Configuration

In [ ]:
import os
import subprocess
import re
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Configuration
MODEL = "facebook/opt-125m"  # Change to opt-1.3b, opt-6.7b, etc. if you have resources
DATASET = "c4"  # Calibration dataset (c4, wikitext2, or ptb)
EVAL_DATASET = "wikitext2"  # Dataset to report perplexity on
DEVICE = "cuda:0"
SPARSITIES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

print(f"Model: {MODEL}")
print(f"Calibration dataset: {DATASET}")
print(f"Evaluation dataset: {EVAL_DATASET}")
print(f"Sparsity levels: {SPARSITIES}")

## 2. Run Dense Baseline

In [ ]:
# First, run the dense (unpruned) model
print("Running dense baseline...")
cmd = f"python opt.py {MODEL} {DATASET}"
print(f"Command: {cmd}")
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## 3. Run SparseGPT Experiments

In [ ]:
# Run SparseGPT at different sparsity levels
sparsegpt_results = {}

for sparsity in SPARSITIES:
    if sparsity == 0.0:
        continue  # Skip 0.0, already done in dense baseline
    
    print(f"\n{'='*60}")
    print(f"Running SparseGPT with sparsity = {sparsity}")
    print('='*60)
    
    cmd = f"python opt.py {MODEL} {DATASET} --sparsity {sparsity}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Parse perplexity from output
    pattern = f"{EVAL_DATASET}.*?Perplexity: ([\d.]+)"
    match = re.search(pattern, result.stdout, re.DOTALL)
    if match:
        ppl = float(match.group(1))
        sparsegpt_results[sparsity] = ppl
        print(f"Perplexity: {ppl:.3f}")
    else:
        print("Could not parse perplexity")
        print(result.stdout[-500:])  # Print last 500 chars

print(f"\nSparseGPT results: {sparsegpt_results}")

## 4. Run Magnitude Pruning Baseline

In [ ]:
# Run magnitude pruning at different sparsity levels
magnitude_results = {}

for sparsity in SPARSITIES:
    if sparsity == 0.0:
        continue
    
    print(f"\n{'='*60}")
    print(f"Running Magnitude Pruning with sparsity = {sparsity}")
    print('='*60)
    
    cmd = f"python opt.py {MODEL} {DATASET} --sparsity {sparsity} --gmp"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Parse perplexity
    pattern = f"{EVAL_DATASET}.*?Perplexity: ([\d.]+)"
    match = re.search(pattern, result.stdout, re.DOTALL)
    if match:
        ppl = float(match.group(1))
        magnitude_results[sparsity] = ppl
        print(f"Perplexity: {ppl:.3f}")
    else:
        print("Could not parse perplexity")
        print(result.stdout[-500:])

print(f"\nMagnitude results: {magnitude_results}")

## 5. Visualize Results

In [ ]:
# Plot the comparison
fig, ax = plt.subplots(figsize=(10, 6))

# Get dense baseline perplexity (should extract from step 2)
# For now, using sparsity=0 or the first result
dense_ppl = 8.0  # Replace with actual value from dense run

# Plot dense baseline
ax.axhline(y=dense_ppl, color='gray', linestyle='--', linewidth=2, label='Dense')

# Plot SparseGPT
if sparsegpt_results:
    sparsities_sg = sorted(sparsegpt_results.keys())
    ppls_sg = [sparsegpt_results[s] for s in sparsities_sg]
    ax.plot(sparsities_sg, ppls_sg, 'o-', color='#1f77b4', 
            linewidth=2, markersize=8, label='SparseGPT')

# Plot Magnitude
if magnitude_results:
    sparsities_mag = sorted(magnitude_results.keys())
    ppls_mag = [magnitude_results[s] for s in sparsities_mag]
    ax.plot(sparsities_mag, ppls_mag, 's-', color='#ff7f0e', 
            linewidth=2, markersize=8, label='Magnitude')

ax.set_xlabel('Sparsity', fontsize=13)
ax.set_ylabel(f'Perplexity on {EVAL_DATASET}', fontsize=13)
ax.set_title(f'Sparsity-vs-perplexity comparison on {MODEL}', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sparsity_vs_perplexity_reproduction.png', dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved to sparsity_vs_perplexity_reproduction.png")

## 6. Results Summary Table

In [ ]:
import pandas as pd

# Create results dataframe
data = []
for sparsity in sorted(set(list(sparsegpt_results.keys()) + list(magnitude_results.keys()))):
    row = {'Sparsity': sparsity}
    if sparsity in sparsegpt_results:
        row['SparseGPT'] = sparsegpt_results[sparsity]
    if sparsity in magnitude_results:
        row['Magnitude'] = magnitude_results[sparsity]
    data.append(row)

df = pd.DataFrame(data)
print("\nResults Summary:")
print("="*60)
print(df.to_string(index=False))
print("="*60)

# Calculate improvement
if SPARSITIES[-1] in sparsegpt_results and SPARSITIES[-1] in magnitude_results:
    max_sparsity = SPARSITIES[-1]
    improvement = ((magnitude_results[max_sparsity] - sparsegpt_results[max_sparsity]) / 
                   magnitude_results[max_sparsity] * 100)
    print(f"\nAt {max_sparsity*100}% sparsity, SparseGPT achieves {improvement:.1f}% lower perplexity than Magnitude pruning")

## Notes

### For OPT-175B Reproduction:
1. Request access from Meta for OPT-175B
2. Convert checkpoint to HuggingFace format
3. Set `MODEL = "path/to/opt-175b"` 
4. Requires significant GPU resources (multiple A100s)

### Key Parameters:
- `--sparsity`: Target sparsity level (0.0 to 1.0)
- `--gmp`: Use magnitude pruning instead of SparseGPT
- `--nsamples`: Number of calibration samples (default: 128)
- `--percdamp`: Hessian dampening factor (default: 0.01)

### Expected Behavior:
SparseGPT should maintain lower perplexity than magnitude pruning at high sparsity levels (>50%), as shown in the paper's Figure 1.